# Data Cleaning

In [2]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from rapidfuzz import fuzz,process
import sys
import duckdb as db
from pathlib import Path
sys.path.append('..\src')
from py_def_class import f1_rule_era,abandoned_lap,practice_quali_new

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
base_path = Path("..") / "data" / "raw" / "2026" / "Spanish GP"

files = [
    "Practice/2026-Spanish Grand Prix-Practice 1.csv",
    "Practice/2026-Spanish Grand Prix-Practice 2.csv",
    "Practice/2026-Spanish Grand Prix-Practice 3.csv",
    "Qualifying/2026-Spanish Grand Prix-Qualifying.csv",
    "Race/2026-Spanish Grand Prix-Race.csv",
    #"Sprint-Qualifying/2026-Canadian Grand Prix-Sprint Qualifying.csv",
    #"Sprint-Race/2026-Canadian Grand Prix-Sprint.csv"
]

driver_team_2026 = {
    "NOR": "McLaren",
    "PIA": "McLaren",
    "VER": "Red Bull Racing",
    "HAD": "Red Bull Racing",
    "LEC": "Ferrari",
    "HAM": "Ferrari",
    "RUS": "Mercedes",
    "ANT": "Mercedes",
    "ALO": "Aston Martin",
    "STR": "Aston Martin",
    "GAS": "Alpine",
    "COL": "Alpine",
    "ALB": "Williams",
    "SAI": "Williams",
    "OCO": "Haas F1 Team",
    "BEA": "Haas F1 Team",
    "LIN": "Racing Bulls",
    "LAW": "Racing Bulls",
    "HUL": "Audi",
    "BOR": "Audi",
    "PER": "Cadillac",
    "BOT": "Cadillac"
}

for file in files:
    file_path = base_path / file
    
    df_clean = pd.read_csv(file_path)
    df_clean["Team"] = df_clean["Driver"].map(driver_team_2026)
    
    df_clean.to_csv(file_path, index=False)
    
    print(f"Saved: {file_path}")

Saved: ..\data\raw\2026\Spanish GP\Practice\2026-Spanish Grand Prix-Practice 1.csv
Saved: ..\data\raw\2026\Spanish GP\Practice\2026-Spanish Grand Prix-Practice 2.csv
Saved: ..\data\raw\2026\Spanish GP\Practice\2026-Spanish Grand Prix-Practice 3.csv
Saved: ..\data\raw\2026\Spanish GP\Qualifying\2026-Spanish Grand Prix-Qualifying.csv
Saved: ..\data\raw\2026\Spanish GP\Race\2026-Spanish Grand Prix-Race.csv


## 1. Practice
### Laptimes

In [10]:
df_p1 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv').iloc[:,1:]
df_p1['Season'] = '2018'
df_p1['Rule_Era'] = df_p1.apply(f1_rule_era,axis=1)
df_p1['Abandoned_Lap'] = df_p1.apply(abandoned_lap,axis=1)
df_p1['GP'] = 'Australian GP'
df_p1['Time'] = pd.to_timedelta(df_p1['Time'])
df_p1['Time_Minutes'] = np.ceil(df_p1['Time'].dt.total_seconds()/60)
df_p1.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,LapTime_in_seconds,laptime_sum_sectortimes,Season,Rule_Era,Abandoned_Lap,GP,Time_Minutes
0,0 days 00:41:09.160000,VAN,2,NaN,1.0,1.0,0 days 00:05:01.044000,0 days 00:06:54.173000,50.217,26.276,NaN,0 days 00:05:47.815000,0 days 00:06:14.097000,NaN,223.0,259.0,NaN,199.0,False,ULTRASOFT,1.0,True,McLaren,0 days 00:05:01.044000,2018-03-23 01:01:01.768,21.0,NaN,False,NaN,False,False,NaN,NaN,2018,High-Downforce,1,Australian GP,42.0
1,0 days 01:01:01.904000,VAN,2,NaN,2.0,2.0,0 days 00:41:09.160000,0 days 00:42:59.399000,38.376,24.973,NaN,0 days 00:41:44.897000,0 days 00:42:09.851000,NaN,256.0,276.0,NaN,194.0,False,ULTRASOFT,2.0,False,McLaren,0 days 00:41:09.160000,2018-03-23 01:37:09.884,1.0,NaN,False,NaN,False,False,NaN,NaN,2018,High-Downforce,1,Australian GP,62.0
2,0 days 01:02:45.996000,VAN,2,0 days 00:01:44.092000,3.0,3.0,0 days 01:01:04.578000,NaN,40.062,25.482,38.548,0 days 01:01:41.966000,0 days 01:02:07.448000,0 days 01:02:45.996000,213.0,274.0,274.0,202.0,False,SOFT,1.0,True,McLaren,0 days 01:01:01.904000,2018-03-23 01:57:02.628,1.0,NaN,False,NaN,False,False,104.092,104.092,2018,High-Downforce,0,Australian GP,63.0
3,0 days 01:04:15.554000,VAN,2,0 days 00:01:29.558000,4.0,3.0,NaN,NaN,29.362,24.068,36.128,0 days 01:03:15.358000,0 days 01:03:39.426000,0 days 01:04:15.554000,263.0,275.0,276.0,292.0,True,SOFT,2.0,True,McLaren,0 days 01:02:45.996000,2018-03-23 01:58:46.720,1.0,NaN,False,NaN,False,True,89.558,89.558,2018,High-Downforce,0,Australian GP,65.0
4,0 days 01:21:30.344000,VAN,2,NaN,5.0,3.0,NaN,0 days 01:05:46.754000,29.632,24.110,NaN,0 days 01:04:45.429000,0 days 01:05:09.322000,NaN,266.0,276.0,NaN,293.0,False,SOFT,3.0,True,McLaren,0 days 01:04:15.554000,2018-03-23 02:00:16.278,12.0,NaN,False,NaN,False,False,NaN,NaN,2018,High-Downforce,1,Australian GP,82.0


In [ ]:
path_list=[]
data_path = Path(r'..\data\raw\2018')

for file in data_path.iterdir():
    if file.is_dir():
        #gp=file.split('\')
        path_list.append(file.name)
path_list = path_list[1:]
path_list

In [ ]:
training_1_time_w_diff = df_p1.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_1_time_w_diff.columns)
loc_cols = list(training_1_time_w_diff.columns)
loc_cols.append('Compound')
training_1_time_w_diff['LapTimeDiff_P1'] = training_1_time_w_diff['laptime_sum_sectortimes'] - training_1_time_w_diff['laptime_sum_sectortimes'].min()
df_p1.loc[:,loc_cols]
p1_fast_time = pd.merge(training_1_time_w_diff,df_p1.loc[:,loc_cols],on=merge_cols,how='left')
if p1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p1_fast_time['laptime_sum_sectortimes'] = p1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p1_fast_time['LapTimeDiff_P1'] = p1_fast_time['LapTimeDiff_P1'].fillna(0.0)
    p1_fast_time['Compound'] = p1_fast_time['Compound'].fillna('No Tyres Used')
p1_fast_time.rename(columns={'laptime_sum_sectortimes':'laptime_sum_sectortimes_P1'},inplace=True)
p1_fast_time

In [ ]:
driver = 'SIR'

fastest_ = df_p1[(df_p1['Driver'].eq(driver))]['laptime_sum_sectortimes'].min()
df_p1[(df_p1['Driver'].eq(driver)) & (df_p1['laptime_sum_sectortimes'].eq(fastest_))]

### Weather

In [ ]:
df_p1_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1-weather.csv').iloc[:,1:]
df_p1_weather['Time'] = pd.to_timedelta(df_p1_weather['Time'])
df_p1_weather['Time_Minutes'] = np.ceil(df_p1_weather['Time'].dt.total_seconds()/60)
df_p1_weather['Rainfall'] = df_p1_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p1_weather.drop(['Time'],axis=1,inplace=True)
df_p1_weather

In [ ]:
df_p1 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv').iloc[:,1:]
df_p1['Season'] = '2018'
df_p1['Rule_Era'] = df_p1.apply(f1_rule_era,axis=1)
df_p1['Abandoned_Lap'] = df_p1.apply(abandoned_lap,axis=1)
df_p1['GP'] = 'Australian GP'
df_p1['Time'] = pd.to_timedelta(df_p1['Time'])
df_p1['Time_Minutes'] = np.ceil(df_p1['Time'].dt.total_seconds()/60)
df_p1 = pd.merge(df_p1,df_p1_weather,on='Time_Minutes',how='inner')
df_p1 = df_p1[df_p1['Abandoned_Lap'].eq(0)]

training_1_time_w_diff = df_p1.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_1_time_w_diff.columns)
loc_cols = list(training_1_time_w_diff.columns)
loc_cols.append('Compound')
training_1_time_w_diff['LapTimeDiff_P1'] = training_1_time_w_diff['laptime_sum_sectortimes'] - training_1_time_w_diff['laptime_sum_sectortimes'].min()
df_p1.loc[:,loc_cols]
p1_fast_time = pd.merge(training_1_time_w_diff,df_p1.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p1_fast_time = p1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)

## 2. Practice
### Laptimes

In [ ]:
df_p2 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv').iloc[:,1:]
df_p2['Season'] = '2018'
df_p2['Rule_Era'] = df_p2.apply(f1_rule_era,axis=1)
df_p2['Abandoned_Lap'] = df_p2.apply(abandoned_lap,axis=1)
df_p2['GP'] = 'Australian GP'
df_p2.head()

In [ ]:
training_2_time_w_diff = df_p2.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_2_time_w_diff.columns)
loc_cols = list(training_2_time_w_diff.columns)
loc_cols.append('Compound')
training_2_time_w_diff['LapTimeDiff_P2'] = training_2_time_w_diff['laptime_sum_sectortimes'] - training_2_time_w_diff['laptime_sum_sectortimes'].min()
df_p2.loc[:,loc_cols]
p2_fast_time = pd.merge(training_2_time_w_diff,df_p2.loc[:,loc_cols],on=merge_cols,how='left')
if p2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p2_fast_time['laptime_sum_sectortimes'] = p2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p2_fast_time['LapTimeDiff_P2'] = p2_fast_time['LapTimeDiff_P2'].fillna(0.0)
    p2_fast_time['Compound'] = p2_fast_time['Compound'].fillna('No Tyres Used')
p2_fast_time.rename(columns={'laptime_sum_sectortimes':'laptime_sum_sectortimes_P2'},inplace=True)
p2_fast_time

### Weather

In [ ]:
df_p2_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2-weather.csv').iloc[:,1:]
df_p2_weather['Time'] = pd.to_timedelta(df_p2_weather['Time'])
df_p2_weather['Time_Minutes'] = np.ceil(df_p2_weather['Time'].dt.total_seconds()/60)
df_p2_weather['Rainfall'] = df_p2_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p2_weather.drop(['Time'],axis=1,inplace=True)
df_p2_weather

In [ ]:
df_p2 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv').iloc[:,1:]
df_p2['Season'] = '2018'
df_p2['Rule_Era'] = df_p2.apply(f1_rule_era,axis=1)
df_p2['Abandoned_Lap'] = df_p2.apply(abandoned_lap,axis=1)
df_p2['GP'] = 'Australian GP'
df_p2['Time'] = pd.to_timedelta(df_p2['Time'])
df_p2['Time_Minutes'] = np.ceil(df_p2['Time'].dt.total_seconds()/60)
df_p2 = pd.merge(df_p2,df_p2_weather,on='Time_Minutes',how='inner')
df_p2 = df_p2[df_p2['Abandoned_Lap'].eq(0)]

training_2_time_w_diff = df_p2.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_2_time_w_diff.columns)
loc_cols = list(training_2_time_w_diff.columns)
loc_cols.append('Compound')
training_2_time_w_diff['LapTimeDiff_P1'] = training_2_time_w_diff['laptime_sum_sectortimes'] - training_2_time_w_diff['laptime_sum_sectortimes'].min()
df_p2.loc[:,loc_cols]
p2_fast_time = pd.merge(training_2_time_w_diff,df_p2.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)

## 3. Practice
### Laptimes

In [ ]:
df_p3 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv').iloc[:,1:]
df_p3['Season'] = '2018'
df_p3['Rule_Era'] = df_p3.apply(f1_rule_era,axis=1)
df_p3['Abandoned_Lap'] = df_p3.apply(abandoned_lap,axis=1)
df_p3['GP'] = 'Australian GP'
df_p3.head()

In [ ]:
training_3_time_w_diff = df_p3.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_3_time_w_diff.columns)
loc_cols = list(training_3_time_w_diff.columns)
loc_cols.append('Compound')
training_3_time_w_diff['LapTimeDiff_P3'] = training_3_time_w_diff['laptime_sum_sectortimes'] - training_3_time_w_diff['laptime_sum_sectortimes'].min()
df_p3.loc[:,loc_cols]
p3_fast_time = pd.merge(training_3_time_w_diff,df_p3.loc[:,loc_cols],on=merge_cols,how='left')
if p3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p3_fast_time['laptime_sum_sectortimes'] = p3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p3_fast_time['LapTimeDiff_P3'] = p3_fast_time['LapTimeDiff_P3'].fillna(0.0)
    p3_fast_time['Compound'] = p3_fast_time['Compound'].fillna('No Tyres Used')
p3_fast_time.rename(columns={'laptime_sum_sectortimes':'laptime_sum_sectortimes_P3'},inplace=True)
p3_fast_time

### Weather

In [ ]:
df_p3_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3-weather.csv').iloc[:,1:]
df_p3_weather['Time'] = pd.to_timedelta(df_p3_weather['Time'])
df_p3_weather['Time_Minutes'] = np.ceil(df_p3_weather['Time'].dt.total_seconds()/60)
df_p3_weather['Rainfall'] = df_p3_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p3_weather.drop(['Time'],axis=1,inplace=True)
df_p3_weather

In [ ]:
df_p2[['Driver','Team','GP']].value_counts().to_frame().reset_index().iloc[:,:-1]

In [ ]:
df_p3 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv').iloc[:,1:]
df_p3['Season'] = '2018'
df_p3['Rule_Era'] = df_p3.apply(f1_rule_era,axis=1)
df_p3['Abandoned_Lap'] = df_p3.apply(abandoned_lap,axis=1)
df_p3['GP'] = 'Australian GP'
df_p3['Time'] = pd.to_timedelta(df_p3['Time'])
df_p3['Time_Minutes'] = np.ceil(df_p3['Time'].dt.total_seconds()/60)
df_p3 = pd.merge(df_p3,df_p3_weather,on='Time_Minutes',how='inner')
df_p3 = df_p3[df_p3['Abandoned_Lap'].eq(0)]


combined = df_p1
for d in [df_p2,df_p3]:
    combined = pd.concat([combined,d],axis=0)
master_driver = combined[['Driver','Team','GP']].value_counts().to_frame().reset_index().iloc[:,:-1]

training_3_time_w_diff = df_p3.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_3_time_w_diff.columns)
loc_cols = list(training_3_time_w_diff.columns)
loc_cols.append('Compound')
training_3_time_w_diff['LapTimeDiff'] = training_3_time_w_diff['laptime_sum_sectortimes'] - training_3_time_w_diff['laptime_sum_sectortimes'].min()
df_p3.loc[:,loc_cols]
p3_fast_time = pd.merge(training_3_time_w_diff,df_p3.loc[:,loc_cols],on=merge_cols,how='left')
if p3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p3_fast_time['laptime_sum_sectortimes'] = p3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p3_fast_time['LapTimeDiff_P3'] = p3_fast_time['LapTimeDiff_P3'].fillna(0.0)
    p3_fast_time['Compound'] = p3_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = p3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p3_fast_time = p3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p3_fast_time = pd.merge(master_driver,p3_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p3_fast_time.columns if c not in group_col]
for pc in practice_col:
    p3_fast_time.rename(columns={pc:f'{pc}_p3'},inplace=True)
p3_fast_time = p3_fast_time.sort_values('laptime_sum_sectortimes_p3',ascending=True).reset_index(drop=True)
p3_fast_time

## Total Trainig Overview

In [ ]:
df_p1 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv').iloc[:,1:]
df_p1['Season'] = '2018'
df_p1['Rule_Era'] = df_p1.apply(f1_rule_era,axis=1)
df_p1['Abandoned_Lap'] = df_p1.apply(abandoned_lap,axis=1)
df_p1['GP'] = 'Australian GP'
df_p1['Time'] = pd.to_timedelta(df_p1['Time'])
df_p1['Time_Minutes'] = np.ceil(df_p1['Time'].dt.total_seconds()/60)
df_p1_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1-weather.csv').iloc[:,1:]
df_p1_weather['Time'] = pd.to_timedelta(df_p1_weather['Time'])
df_p1_weather['Time_Minutes'] = np.ceil(df_p1_weather['Time'].dt.total_seconds()/60)
df_p1_weather['Rainfall'] = df_p1_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p1_weather.drop(['Time'],axis=1,inplace=True)
df_p1 = pd.merge(df_p1,df_p1_weather,on='Time_Minutes',how='inner')
df_p1 = df_p1[df_p1['Abandoned_Lap'].eq(0)]

df_p2 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv').iloc[:,1:]
df_p2['Season'] = '2018'
df_p2['Rule_Era'] = df_p2.apply(f1_rule_era,axis=1)
df_p2['Abandoned_Lap'] = df_p2.apply(abandoned_lap,axis=1)
df_p2['GP'] = 'Australian GP'
df_p2['Time'] = pd.to_timedelta(df_p2['Time'])
df_p2['Time_Minutes'] = np.ceil(df_p2['Time'].dt.total_seconds()/60)
df_p2_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2-weather.csv').iloc[:,1:]
df_p2_weather['Time'] = pd.to_timedelta(df_p2_weather['Time'])
df_p2_weather['Time_Minutes'] = np.ceil(df_p2_weather['Time'].dt.total_seconds()/60)
df_p2_weather['Rainfall'] = df_p2_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p2_weather.drop(['Time'],axis=1,inplace=True)
df_p2 = pd.merge(df_p2,df_p2_weather,on='Time_Minutes',how='inner')
df_p2 = df_p2[df_p2['Abandoned_Lap'].eq(0)]

df_p3 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv').iloc[:,1:]
df_p3['Season'] = '2018'
df_p3['Rule_Era'] = df_p3.apply(f1_rule_era,axis=1)
df_p3['Abandoned_Lap'] = df_p3.apply(abandoned_lap,axis=1)
df_p3['GP'] = 'Australian GP'
df_p3['Time'] = pd.to_timedelta(df_p3['Time'])
df_p3['Time_Minutes'] = np.ceil(df_p3['Time'].dt.total_seconds()/60)
df_p3_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3-weather.csv').iloc[:,1:]
df_p3_weather['Time'] = pd.to_timedelta(df_p3_weather['Time'])
df_p3_weather['Time_Minutes'] = np.ceil(df_p3_weather['Time'].dt.total_seconds()/60)
df_p3_weather['Rainfall'] = df_p3_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p3_weather.drop(['Time'],axis=1,inplace=True)
df_p3 = pd.merge(df_p3,df_p3_weather,on='Time_Minutes',how='inner')
df_p3 = df_p3[df_p3['Abandoned_Lap'].eq(0)]

combined = df_p1
for d in [df_p2,df_p3]:
    combined = pd.concat([combined,d],axis=0)
master_driver = combined[['Driver','Team','GP']].value_counts().to_frame().reset_index().iloc[:,:-1]

#df_training_overview = pd.DataFrame()

#practice 1
training_1_time_w_diff = df_p1.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_1_time_w_diff.columns)
loc_cols = list(training_1_time_w_diff.columns)
loc_cols.append('Compound')
training_1_time_w_diff['LapTimeDiff'] = training_1_time_w_diff['laptime_sum_sectortimes'] - training_1_time_w_diff['laptime_sum_sectortimes'].min()
df_p1.loc[:,loc_cols]
p1_fast_time = pd.merge(training_1_time_w_diff,df_p1.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p1_fast_time = p1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p1_fast_time = pd.merge(master_driver,p1_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p1_fast_time.columns if c not in group_col]
for pc in practice_col:
    p1_fast_time.rename(columns={pc:f'{pc}_p1'},inplace=True)
p1_fast_time = p1_fast_time.sort_values('laptime_sum_sectortimes_p1',ascending=True).reset_index(drop=True)

#practice 2
training_2_time_w_diff = df_p2.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_2_time_w_diff.columns)
loc_cols = list(training_2_time_w_diff.columns)
loc_cols.append('Compound')
training_2_time_w_diff['LapTimeDiff'] = training_2_time_w_diff['laptime_sum_sectortimes'] - training_2_time_w_diff['laptime_sum_sectortimes'].min()
df_p2.loc[:,loc_cols]
p2_fast_time = pd.merge(training_2_time_w_diff,df_p2.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p2_fast_time = p2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p2_fast_time = pd.merge(master_driver,p2_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p2_fast_time.columns if c not in group_col]
for pc in practice_col:
    p2_fast_time.rename(columns={pc:f'{pc}_p2'},inplace=True)
p2_fast_time = p2_fast_time.sort_values('laptime_sum_sectortimes_p2',ascending=True).reset_index(drop=True)

#practice 3
training_3_time_w_diff = df_p3.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_3_time_w_diff.columns)
loc_cols = list(training_3_time_w_diff.columns)
loc_cols.append('Compound')
training_3_time_w_diff['LapTimeDiff'] = training_3_time_w_diff['laptime_sum_sectortimes'] - training_3_time_w_diff['laptime_sum_sectortimes'].min()
df_p3.loc[:,loc_cols]
p3_fast_time = pd.merge(training_3_time_w_diff,df_p3.loc[:,loc_cols],on=merge_cols,how='left')
if p3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p3_fast_time['laptime_sum_sectortimes'] = p3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p3_fast_time['LapTimeDiff_P3'] = p3_fast_time['LapTimeDiff_P3'].fillna(0.0)
    p3_fast_time['Compound'] = p3_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = p3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p3_fast_time = p3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p3_fast_time = pd.merge(master_driver,p3_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p3_fast_time.columns if c not in group_col]
for pc in practice_col:
    p3_fast_time.rename(columns={pc:f'{pc}_p3'},inplace=True)
p3_fast_time = p3_fast_time.sort_values('laptime_sum_sectortimes_p3',ascending=True).reset_index(drop=True)

#practice overview
trainings = [p2_fast_time,p3_fast_time]
df_training_overview = p1_fast_time.copy()
for t in trainings:
    df_training_overview = pd.merge(df_training_overview,t,on=['Driver','Team','GP','Rule_Era'],how='left')
df_training_overview['Season'] = 2018
df_training_overview

## Qualifying
### Laptimes

In [ ]:
df_quali = pd.read_csv(r'..\data\raw\2018\Bahrain GP\Qualifying\2018-Bahrain Grand Prix-Qualifying.csv').iloc[:,1:]
df_quali['Season'] = '2018'
df_quali['Rule_Era'] = df_quali.apply(f1_rule_era,axis=1)
df_quali['Abandoned_Lap'] = df_quali.apply(abandoned_lap,axis=1)
df_quali['GP'] = 'Australian GP'
df_quali['Time'] = pd.to_timedelta(df_quali['Time'])
df_quali['Time_Minutes'] = np.ceil(df_quali['Time'].dt.total_seconds()/60)
df_quali.head()

In [ ]:
np.sort(list(df_quali['Time'].unique()))

In [ ]:
quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
quali_time_df.head()

In [ ]:
quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
quali_time_df['Time_Minutes'] = np.ceil(quali_time_df['Time'].dt.total_seconds()/60)
#next getting the the indexies; after that, use the index and subtact one from it, getting the string value for minutes and turn it back to int and use it with pd.Timedelta()
quali_time_df[((quali_time_df['Time_Diff']>=5) & (quali_time_df['Time_Minutes']>=18)) | ((quali_time_df['Time_Diff']>=6) & (quali_time_df['Time_Minutes']>=33))]
#str(quali_time_df.iloc[134,0]).split(':')[1]

In [ ]:
df_quali_session = quali_time_df[((quali_time_df['Time_Diff']>=5) & (quali_time_df['Time_Minutes']>=18)) | ((quali_time_df['Time_Diff']>=6) & (quali_time_df['Time_Minutes']>=33))].copy()
#df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session = df_quali_session['Time_Minutes'].to_frame().reset_index(drop=True)
df_quali_session.loc[2] = quali_time_df['Time_Minutes'].max()
df_quali_session

In [ ]:
#quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=6].index]
#df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
#df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
#df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])
df_quali_session = quali_time_df[((quali_time_df['Time_Diff']>=5) & (quali_time_df['Time_Minutes']>=18)) | ((quali_time_df['Time_Diff']>=6) & (quali_time_df['Time_Minutes']>=33))].copy()
###df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session = df_quali_session['Time_Minutes'].to_frame().reset_index(drop=True)

df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][0])]
quali_1_time_w_diff = df_q1.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q1_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q1_fast_time['laptime_sum_sectortimes'] = q1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q1_fast_time['LapTimeDiff'] = q1_fast_time['LapTimeDiff'].fillna(0.0)
    q1_fast_time['Compound'] = q1_fast_time['Compound'].fillna('No Tyres Used')
#q1_fast_time.rename(columns={'laptime_sum_sectortimes':'laptime_sum_sectortimes_P3','Compound':'Compound_practice_3'},inplace=True)
driver_q1 = q1_fast_time[['Driver','Team','GP']].value_counts().to_frame().reset_index()
q1_drop = q1_fast_time.iloc[-5:,:]
q1_drop

In [ ]:
driver_q1

In [ ]:
#quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=6].index]
#df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
#df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
#df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])
df_quali_session = quali_time_df[((quali_time_df['Time_Diff']>=5) & (quali_time_df['Time_Minutes']>=18)) | ((quali_time_df['Time_Diff']>=6) & (quali_time_df['Time_Minutes']>=33))].copy()
###df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session = df_quali_session['Time_Minutes'].to_frame().reset_index(drop=True)

df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][1])]
quali_1_time_w_diff = df_q1.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q2_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q2_fast_time['laptime_sum_sectortimes'] = q2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q2_fast_time['LapTimeDiff'] = q2_fast_time['LapTimeDiff'].fillna(0.0)
    q2_fast_time['Compound'] = q2_fast_time['Compound'].fillna('No Tyres Used')

#here we drop the drivers from q1
drivers_dropped_q1 = list(q1_drop['Driver'])
q2_fast_time = q2_fast_time[~q2_fast_time['Driver'].isin(drivers_dropped_q1)]
driver_q2 = q2_fast_time[['Driver','Team','GP']].value_counts().to_frame().reset_index()
driver_q2 = pd.concat([driver_q1,driver_q2],axis=0)
q2_drop = q2_fast_time.iloc[-5:,:]
q2_drop

In [ ]:
driver_q2

In [ ]:
#quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=6].index]
#df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
#df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
#df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])
df_quali_session = quali_time_df[((quali_time_df['Time_Diff']>=5) & (quali_time_df['Time_Minutes']>=18)) | ((quali_time_df['Time_Diff']>=6) & (quali_time_df['Time_Minutes']>=33))].copy()
df_quali_session.loc[2] = quali_time_df['Time_Minutes'].max()
df_quali_session = df_quali_session['Time_Minutes'].to_frame().reset_index(drop=True)

df_q3 = df_quali[df_quali['Time']>=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][2])]
quali_3_time_w_diff = df_q3.groupby(['Driver','Team','GP','Rule_Era'])['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_3_time_w_diff.columns)
loc_cols = list(quali_3_time_w_diff.columns)
loc_cols.append('Compound')
quali_3_time_w_diff['LapTimeDiff'] = quali_3_time_w_diff['laptime_sum_sectortimes'] - quali_3_time_w_diff['laptime_sum_sectortimes'].min()
df_q3.loc[:,loc_cols]
q3_fast_time = pd.merge(quali_3_time_w_diff,df_q3.loc[:,loc_cols],on=merge_cols,how='left')
if q3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q3_fast_time['laptime_sum_sectortimes'] = q3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q3_fast_time['LapTimeDiff'] = q3_fast_time['LapTimeDiff'].fillna(0.0)
    q3_fast_time['Compound'] = q3_fast_time['Compound'].fillna('No Tyres Used')
#here we drop the drivers from q1 and q2
drivers_dropped_q1_a_q2 = drivers_dropped_q1 + list(q2_drop['Driver'])
q3_fast_time = q3_fast_time[~q3_fast_time['Driver'].isin(drivers_dropped_q1_a_q2)]
driver_q3 = q3_fast_time[['Driver','Team','GP']].value_counts().to_frame().reset_index()
driver_q3 = pd.concat([driver_q2,driver_q3],axis=0)
q3_fast_time

In [ ]:
session_app = driver_q3.groupby(['Driver','Team'])['count'].sum().reset_index()
session_app.rename(columns={'count':'Session'},inplace=True)
session_app['Session'] = session_app['Session'].apply(lambda x: 'Q1' if x == 1 else ('Q2' if x == 2 else 'Q3'))
session_app

In [ ]:
df_quali_overview = pd.DataFrame()

quali_sessions = [q1_drop,q2_drop,q3_fast_time]

for q in quali_sessions:
    df_quali_overview = pd.concat([df_quali_overview,q],axis=0)

session_app = driver_q3.groupby(['Driver','Team'])['count'].sum().reset_index()
session_app.rename(columns={'count':'Session'},inplace=True)
session_app['Session'] = session_app['Session'].apply(lambda x: 'Q1' if x == 1 else ('Q2' if x == 2 else 'Q3'))

df_quali_overview = pd.merge(session_app,df_quali_overview,on=['Driver','Team'],how='left')

q1_missing_df = df_quali_overview[(df_quali_overview['laptime_sum_sectortimes'].isna()) & (df_quali_overview['Session'].eq('Q1'))]
q2_missing_df = df_quali_overview[(df_quali_overview['laptime_sum_sectortimes'].isna()) & (df_quali_overview['Session'].eq('Q2'))]

if len(q1_missing_df) > 0:
    q1_add = q1_fast_time[(q1_fast_time['Driver'].isin(list(q1_missing_df['Driver']))) & (q1_fast_time['Team'].isin(list(q1_missing_df['Team'])))]
    q1_add['Session'] = 'Q1'

if len(q2_missing_df) > 0:
    q2_add = q2_fast_time[(q2_fast_time['Driver'].isin(list(q2_missing_df['Driver']))) & (q2_fast_time['Team'].isin(list(q2_missing_df['Team'])))]
    q2_add['Session'] = 'Q2'

fill_df = pd.concat([q1_add, q2_add], axis=0, ignore_index=True)

cols_to_fill = ['GP', 'Rule_Era', 'laptime_sum_sectortimes', 'LapTimeDiff', 'Compound']

df_quali_overview = df_quali_overview.merge(
    fill_df[['Driver', 'Team', 'Session'] + cols_to_fill],
    on=['Driver', 'Team', 'Session'],
    how='left',
    suffixes=('', '_fill')
)

for col in cols_to_fill:
    df_quali_overview[col] = df_quali_overview[col].fillna(df_quali_overview[f'{col}_fill'])

df_quali_overview.drop(columns=[f'{col}_fill' for col in cols_to_fill], inplace=True)
group_col = ['Driver','Team','GP','Rule_Era','Session']
quali_col = [c for c in df_quali_overview.columns if c not in group_col]
for qc in quali_col:
    df_quali_overview.rename(columns={qc:f'{qc}_quali'},inplace=True)
df_quali_overview.sort_values('laptime_sum_sectortimes_quali').reset_index(drop=True)

### Weather

In [ ]:
df_quali_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying-weather.csv').iloc[:,1:]
df_quali_weather['Time'] = pd.to_timedelta(df_quali_weather['Time'])
df_quali_weather['Time_Minutes'] = np.ceil(df_quali_weather['Time'].dt.total_seconds()/60)
df_quali_weather['Rainfall'] = df_quali_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_quali_weather.drop(['Time'],axis=1,inplace=True)
df_quali_weather

In [ ]:
#combining laptime with weather data
df_quali = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying.csv').iloc[:,1:]
df_quali['Season'] = '2018'
df_quali['Rule_Era'] = df_quali.apply(f1_rule_era,axis=1)
df_quali['Abandoned_Lap'] = df_quali.apply(abandoned_lap,axis=1)
df_quali['GP'] = 'Australian GP'
df_quali['Time'] = pd.to_timedelta(df_quali['Time'])
df_quali['Time_Minutes'] = np.ceil(df_quali['Time'].dt.total_seconds()/60)
df_quali_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying-weather.csv').iloc[:,1:]
df_quali_weather['Time'] = pd.to_timedelta(df_quali_weather['Time'])
df_quali_weather['Time_Minutes'] = np.ceil(df_quali_weather['Time'].dt.total_seconds()/60)
df_quali_weather['Rainfall'] = df_quali_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_quali_weather.drop(['Time'],axis=1,inplace=True)
df_quali = pd.merge(df_quali,df_quali_weather,on='Time_Minutes',how='inner')
df_quali = df_quali[df_quali['Abandoned_Lap'].eq(0)]


#creating the code block which allows us to divide into Q1, Q2 and Q3
quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
quali_time_df['Time_Minutes'] = np.ceil(quali_time_df['Time'].dt.total_seconds()/60)
quali_time_df[quali_time_df['Time_Diff']>=5]
#cutoff time for each qualifiying session
quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=5].index]
df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])


#Q1 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][0])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q1_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q1_fast_time['laptime_sum_sectortimes'] = q1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q1_fast_time['LapTimeDiff'] = q1_fast_time['LapTimeDiff'].fillna(0.0)
    q1_fast_time['Compound'] = q1_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = q1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q1_fast_time = q1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
q1_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][0]
q1_fast_time = q1_fast_time.iloc[-5:,:]


#Q2 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][1])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q2_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q2_fast_time['laptime_sum_sectortimes'] = q2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q2_fast_time['LapTimeDiff'] = q2_fast_time['LapTimeDiff'].fillna(0.0)
    q2_fast_time['Compound'] = q2_fast_time['Compound'].fillna('No Tyres Used')
q2_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][1]
fastest_lap_idx = q2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q2_fast_time = q2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1
drivers_dropped_q1 = list(q1_fast_time['Driver'])
q2_fast_time = q2_fast_time[~q2_fast_time['Driver'].isin(drivers_dropped_q1)]
q2_fast_time = q2_fast_time.iloc[-5:,:]


#Q3 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][2])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q3_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q3_fast_time['laptime_sum_sectortimes'] = q3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q3_fast_time['LapTimeDiff'] = q3_fast_time['LapTimeDiff'].fillna(0.0)
    q3_fast_time['Compound'] = q3_fast_time['Compound'].fillna('No Tyres Used')
q3_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][2]
fastest_lap_idx = q3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q3_fast_time = q3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1 and q2
drivers_dropped_q1_a_q2 = drivers_dropped_q1 + list(q2_fast_time['Driver'])
q3_fast_time = q3_fast_time[~q3_fast_time['Driver'].isin(drivers_dropped_q1_a_q2)]
q3_fast_time

In [ ]:
#combining laptime with weather data
df_quali = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying.csv').iloc[:,1:]
df_quali['Season'] = '2018'
df_quali['Rule_Era'] = df_quali.apply(f1_rule_era,axis=1)
df_quali['Abandoned_Lap'] = df_quali.apply(abandoned_lap,axis=1)
df_quali['GP'] = 'Australian GP'
df_quali['Time'] = pd.to_timedelta(df_quali['Time'])
df_quali['Time_Minutes'] = np.ceil(df_quali['Time'].dt.total_seconds()/60)
df_quali_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying-weather.csv').iloc[:,1:]
df_quali_weather['Time'] = pd.to_timedelta(df_quali_weather['Time'])
df_quali_weather['Time_Minutes'] = np.ceil(df_quali_weather['Time'].dt.total_seconds()/60)
df_quali_weather['Rainfall'] = df_quali_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_quali_weather.drop(['Time'],axis=1,inplace=True)
df_quali = pd.merge(df_quali,df_quali_weather,on='Time_Minutes',how='inner')
df_quali = df_quali[df_quali['Abandoned_Lap'].eq(0)]


#creating the code block which allows us to divide into Q1, Q2 and Q3
quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
quali_time_df['Time_Minutes'] = np.ceil(quali_time_df['Time'].dt.total_seconds()/60)
quali_time_df[quali_time_df['Time_Diff']>=5]
#cutoff time for each qualifiying session
quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=5].index]
df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])


#Q1 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][0])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q1_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q1_fast_time['laptime_sum_sectortimes'] = q1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q1_fast_time['LapTimeDiff'] = q1_fast_time['LapTimeDiff'].fillna(0.0)
    q1_fast_time['Compound'] = q1_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = q1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q1_fast_time = q1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
q1_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][0]
q1_fast_time = q1_fast_time.iloc[-5:,:]


#Q2 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][1])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q2_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q2_fast_time['laptime_sum_sectortimes'] = q2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q2_fast_time['LapTimeDiff'] = q2_fast_time['LapTimeDiff'].fillna(0.0)
    q2_fast_time['Compound'] = q2_fast_time['Compound'].fillna('No Tyres Used')
q2_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][1]
fastest_lap_idx = q2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q2_fast_time = q2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1
drivers_dropped_q1 = list(q1_fast_time['Driver'])
q2_fast_time = q2_fast_time[~q2_fast_time['Driver'].isin(drivers_dropped_q1)]
q2_fast_time = q2_fast_time.iloc[-5:,:]


#Q3 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][2])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q3_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q3_fast_time['laptime_sum_sectortimes'] = q3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q3_fast_time['LapTimeDiff'] = q3_fast_time['LapTimeDiff'].fillna(0.0)
    q3_fast_time['Compound'] = q3_fast_time['Compound'].fillna('No Tyres Used')
q3_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][2]
fastest_lap_idx = q3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q3_fast_time = q3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1 and q2
drivers_dropped_q1_a_q2 = drivers_dropped_q1 + list(q2_fast_time['Driver'])
q3_fast_time = q3_fast_time[~q3_fast_time['Driver'].isin(drivers_dropped_q1_a_q2)]


#total qualifiying overview
df_quali_overview = pd.DataFrame()
quali_sessions = [q1_fast_time,q2_fast_time,q3_fast_time]
for q in quali_sessions:
    df_quali_overview = pd.concat([df_quali_overview,q],axis=0)
df_quali_overview = df_quali_overview.sort_index()
group_col = ['Driver','Team','GP','Rule_Era','Session']
quali_col = [c for c in df_quali_overview.columns if c not in group_col]
for qc in quali_col:
    df_quali_overview.rename(columns={qc:f'{qc}_quali'},inplace=True)
df_quali_overview

## Sprint-Qualifying

### Sprint-Qualifying from 2021 - 2022

In 2021, the sprint-race was used to get the grid order, for the sunday race.

In [ ]:
df_2021_brit = pd.DataFrame()

prac_1 = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 1.csv")
prac_2 = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 2.csv")
prac_3 = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 3.csv")
quali = Path(r"..\data\raw\2021\British GP\Qualifying\2021-British Grand Prix-Qualifying.csv")
sprint_quali = Path(r"..\data\raw\2021\British GP\Sprint-Qualifying\2021-British Grand Prix-Qualifying.csv")

prac_1_weather = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 1-weather.csv")
prac_2_weather = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 2-weather.csv")
prac_3_weather = Path(r"..\data\raw\2021\British GP\Practice\2021-British Grand Prix-Practice 3-weather.csv")
quali_weather = Path(r"..\data\raw\2021\British GP\Qualifying\2021-British Grand Prix-Qualifying-weather.csv")
sprint_quali_weather = Path(r"..\data\raw\2021\British GP\Sprint-Qualifying\2021-British Grand Prix-Qualifying-weather.csv")

df_loop = practice_quali_new(
    season=2021,
    gp='British GP',
    timing_paths=[prac_1, prac_2, prac_3, quali, sprint_quali],
    weather_paths=[prac_1_weather, prac_2_weather, prac_3_weather, quali_weather, sprint_quali_weather]
)

df_2021_brit = pd.concat([df_2021_brit, df_loop], ignore_index=True)
df_2021_brit

### Sprint-Qualifiying from 2023 till now

In [ ]:
df_2023_aza = pd.DataFrame()

prac_1 = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 1.csv")
prac_2 = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 2.csv")
prac_3 = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 3.csv")
quali = Path(r"..\data\raw\2023\Azerbaijan GP\Qualifying\2023-Azerbaijan Grand Prix-Qualifying.csv")
sprint_quali = Path(r"..\data\raw\2023\Azerbaijan GP\Sprint-Qualifying\2023-Azerbaijan Grand Prix-Sprint Shootout.csv")

prac_1_weather = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 1-weather.csv")
prac_2_weather = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 2-weather.csv")
prac_3_weather = Path(r"..\data\raw\2023\Azerbaijan GP\Practice\2023-Azerbaijan Grand Prix-Practice 3-weather.csv")
quali_weather = Path(r"..\data\raw\2023\Azerbaijan GP\Qualifying\2023-Azerbaijan Grand Prix-Qualifying-weather.csv")
sprint_quali_weather = Path(r"..\data\raw\2023\Azerbaijan GP\Sprint-Qualifying\2023-Azerbaijan Grand Prix-Sprint Shootout-weather.csv")

df_loop = practice_quali_new(
    season=2023,
    gp='Azerbaijan GP',
    timing_paths=[prac_1, prac_2, prac_3, quali,sprint_quali],
    weather_paths=[prac_1_weather, prac_2_weather, prac_3_weather, quali_weather,sprint_quali_weather]
)

df_2023_aza = pd.concat([df_2023_aza, df_loop], ignore_index=True)
df_2023_aza

## Final GP Practice and Qualifying Time

In [ ]:
#practice 1, 2 and 3
df_p1 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv').iloc[:,1:]
df_p1['Season'] = '2018'
df_p1['Rule_Era'] = df_p1.apply(f1_rule_era,axis=1)
df_p1['Abandoned_Lap'] = df_p1.apply(abandoned_lap,axis=1)
df_p1['GP'] = 'Australian GP'
df_p1['Time'] = pd.to_timedelta(df_p1['Time'])
df_p1['Time_Minutes'] = np.ceil(df_p1['Time'].dt.total_seconds()/60)
df_p1_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1-weather.csv').iloc[:,1:]
df_p1_weather['Time'] = pd.to_timedelta(df_p1_weather['Time'])
df_p1_weather['Time_Minutes'] = np.ceil(df_p1_weather['Time'].dt.total_seconds()/60)
df_p1_weather['Rainfall'] = df_p1_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p1_weather.drop(['Time'],axis=1,inplace=True)
df_p1 = pd.merge(df_p1,df_p1_weather,on='Time_Minutes',how='inner')
df_p1 = df_p1[df_p1['Abandoned_Lap'].eq(0)]

df_p2 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv').iloc[:,1:]
df_p2['Season'] = '2018'
df_p2['Rule_Era'] = df_p2.apply(f1_rule_era,axis=1)
df_p2['Abandoned_Lap'] = df_p2.apply(abandoned_lap,axis=1)
df_p2['GP'] = 'Australian GP'
df_p2['Time'] = pd.to_timedelta(df_p2['Time'])
df_p2['Time_Minutes'] = np.ceil(df_p2['Time'].dt.total_seconds()/60)
df_p2_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2-weather.csv').iloc[:,1:]
df_p2_weather['Time'] = pd.to_timedelta(df_p2_weather['Time'])
df_p2_weather['Time_Minutes'] = np.ceil(df_p2_weather['Time'].dt.total_seconds()/60)
df_p2_weather['Rainfall'] = df_p2_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p2_weather.drop(['Time'],axis=1,inplace=True)
df_p2 = pd.merge(df_p2,df_p2_weather,on='Time_Minutes',how='inner')
df_p2 = df_p2[df_p2['Abandoned_Lap'].eq(0)]

df_p3 = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv').iloc[:,1:]
df_p3['Season'] = '2018'
df_p3['Rule_Era'] = df_p3.apply(f1_rule_era,axis=1)
df_p3['Abandoned_Lap'] = df_p3.apply(abandoned_lap,axis=1)
df_p3['GP'] = 'Australian GP'
df_p3['Time'] = pd.to_timedelta(df_p3['Time'])
df_p3['Time_Minutes'] = np.ceil(df_p3['Time'].dt.total_seconds()/60)
df_p3_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3-weather.csv').iloc[:,1:]
df_p3_weather['Time'] = pd.to_timedelta(df_p3_weather['Time'])
df_p3_weather['Time_Minutes'] = np.ceil(df_p3_weather['Time'].dt.total_seconds()/60)
df_p3_weather['Rainfall'] = df_p3_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_p3_weather.drop(['Time'],axis=1,inplace=True)
df_p3 = pd.merge(df_p3,df_p3_weather,on='Time_Minutes',how='inner')
df_p3 = df_p3[df_p3['Abandoned_Lap'].eq(0)]

combined = df_p1
for d in [df_p2,df_p3]:
    combined = pd.concat([combined,d],axis=0)
master_driver = combined[['Driver','Team','GP']].value_counts().to_frame().reset_index().iloc[:,:-1]

#df_training_overview = pd.DataFrame()

#practice 1
training_1_time_w_diff = df_p1.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_1_time_w_diff.columns)
loc_cols = list(training_1_time_w_diff.columns)
loc_cols.append('Compound')
training_1_time_w_diff['LapTimeDiff'] = training_1_time_w_diff['laptime_sum_sectortimes'] - training_1_time_w_diff['laptime_sum_sectortimes'].min()
df_p1.loc[:,loc_cols]
p1_fast_time = pd.merge(training_1_time_w_diff,df_p1.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p1_fast_time = p1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p1_fast_time = pd.merge(master_driver,p1_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p1_fast_time.columns if c not in group_col]
for pc in practice_col:
    p1_fast_time.rename(columns={pc:f'{pc}_p1'},inplace=True)
p1_fast_time = p1_fast_time.sort_values('laptime_sum_sectortimes_p1',ascending=True).reset_index(drop=True)

#practice 2
training_2_time_w_diff = df_p2.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_2_time_w_diff.columns)
loc_cols = list(training_2_time_w_diff.columns)
loc_cols.append('Compound')
training_2_time_w_diff['LapTimeDiff'] = training_2_time_w_diff['laptime_sum_sectortimes'] - training_2_time_w_diff['laptime_sum_sectortimes'].min()
df_p2.loc[:,loc_cols]
p2_fast_time = pd.merge(training_2_time_w_diff,df_p2.loc[:,loc_cols],on=merge_cols,how='left')
fastest_lap_idx = p2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p2_fast_time = p2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p2_fast_time = pd.merge(master_driver,p2_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p2_fast_time.columns if c not in group_col]
for pc in practice_col:
    p2_fast_time.rename(columns={pc:f'{pc}_p2'},inplace=True)
p2_fast_time = p2_fast_time.sort_values('laptime_sum_sectortimes_p2',ascending=True).reset_index(drop=True)

#practice 3
training_3_time_w_diff = df_p3.groupby(
    #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(training_3_time_w_diff.columns)
loc_cols = list(training_3_time_w_diff.columns)
loc_cols.append('Compound')
training_3_time_w_diff['LapTimeDiff'] = training_3_time_w_diff['laptime_sum_sectortimes'] - training_3_time_w_diff['laptime_sum_sectortimes'].min()
df_p3.loc[:,loc_cols]
p3_fast_time = pd.merge(training_3_time_w_diff,df_p3.loc[:,loc_cols],on=merge_cols,how='left')
if p3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    p3_fast_time['laptime_sum_sectortimes'] = p3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    p3_fast_time['LapTimeDiff_P3'] = p3_fast_time['LapTimeDiff_P3'].fillna(0.0)
    p3_fast_time['Compound'] = p3_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = p3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
p3_fast_time = p3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
p3_fast_time = pd.merge(master_driver,p3_fast_time,on=['Driver','Team','GP'],how='left')
group_col = ['Driver','Team','GP','Rule_Era']
practice_col = [c for c in p3_fast_time.columns if c not in group_col]
for pc in practice_col:
    p3_fast_time.rename(columns={pc:f'{pc}_p3'},inplace=True)
p3_fast_time = p3_fast_time.sort_values('laptime_sum_sectortimes_p3',ascending=True).reset_index(drop=True)

#practice overview
trainings = [p2_fast_time,p3_fast_time]
df_training_overview = p1_fast_time.copy()
for t in trainings:
    df_training_overview = pd.merge(df_training_overview,t,on=['Driver','Team','GP','Rule_Era'],how='left')


#Q1, Q2 and Q3
#combining laptime with weather data
df_quali = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying.csv').iloc[:,1:]
df_quali['Season'] = '2018'
df_quali['Rule_Era'] = df_quali.apply(f1_rule_era,axis=1)
df_quali['Abandoned_Lap'] = df_quali.apply(abandoned_lap,axis=1)
df_quali['GP'] = 'Australian GP'
df_quali['Time'] = pd.to_timedelta(df_quali['Time'])
df_quali['Time_Minutes'] = np.ceil(df_quali['Time'].dt.total_seconds()/60)
df_quali_weather = pd.read_csv(r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying-weather.csv').iloc[:,1:]
df_quali_weather['Time'] = pd.to_timedelta(df_quali_weather['Time'])
df_quali_weather['Time_Minutes'] = np.ceil(df_quali_weather['Time'].dt.total_seconds()/60)
df_quali_weather['Rainfall'] = df_quali_weather['Rainfall'].apply(lambda x: 1 if x else 0)
df_quali_weather.drop(['Time'],axis=1,inplace=True)
df_quali = pd.merge(df_quali,df_quali_weather,on='Time_Minutes',how='inner')
df_quali = df_quali[df_quali['Abandoned_Lap'].eq(0)]


#creating the code block which allows us to divide into Q1, Q2 and Q3
quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
quali_time_df['Time_Minutes'] = np.ceil(quali_time_df['Time'].dt.total_seconds()/60)
quali_time_df[quali_time_df['Time_Diff']>=5]
#cutoff time for each qualifiying session
quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=5].index]
df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])


#Q1 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][0])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q1_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q1_fast_time['laptime_sum_sectortimes'] = q1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q1_fast_time['LapTimeDiff'] = q1_fast_time['LapTimeDiff'].fillna(0.0)
    q1_fast_time['Compound'] = q1_fast_time['Compound'].fillna('No Tyres Used')
fastest_lap_idx = q1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q1_fast_time = q1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
q1_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][0]
q1_fast_time = q1_fast_time.iloc[-5:,:]


#Q2 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][1])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q2_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q2_fast_time['laptime_sum_sectortimes'] = q2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q2_fast_time['LapTimeDiff'] = q2_fast_time['LapTimeDiff'].fillna(0.0)
    q2_fast_time['Compound'] = q2_fast_time['Compound'].fillna('No Tyres Used')
q2_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][1]
fastest_lap_idx = q2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q2_fast_time = q2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1
drivers_dropped_q1 = list(q1_fast_time['Driver'])
q2_fast_time = q2_fast_time[~q2_fast_time['Driver'].isin(drivers_dropped_q1)]
q2_fast_time = q2_fast_time.iloc[-5:,:]


#Q3 laptimes
df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][2])]
quali_1_time_w_diff = df_q1.groupby(
    ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
)['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
merge_cols = list(quali_1_time_w_diff.columns)
loc_cols = list(quali_1_time_w_diff.columns)
loc_cols.append('Compound')
quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
df_q1.loc[:,loc_cols]
q3_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
if q3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
    q3_fast_time['laptime_sum_sectortimes'] = q3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
    q3_fast_time['LapTimeDiff'] = q3_fast_time['LapTimeDiff'].fillna(0.0)
    q3_fast_time['Compound'] = q3_fast_time['Compound'].fillna('No Tyres Used')
q3_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][2]
fastest_lap_idx = q3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
q3_fast_time = q3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
#here we drop the drivers from q1 and q2
drivers_dropped_q1_a_q2 = drivers_dropped_q1 + list(q2_fast_time['Driver'])
q3_fast_time = q3_fast_time[~q3_fast_time['Driver'].isin(drivers_dropped_q1_a_q2)]


#total qualifiying overview
df_quali_overview = pd.DataFrame()
quali_sessions = [q1_fast_time,q2_fast_time,q3_fast_time]
for q in quali_sessions:
    df_quali_overview = pd.concat([df_quali_overview,q],axis=0)
df_quali_overview = df_quali_overview.sort_index()
group_col = ['Driver','Team','GP','Rule_Era','Quali-Session']
quali_col = [c for c in df_quali_overview.columns if c not in group_col]
for qc in quali_col:
    df_quali_overview.rename(columns={qc:f'{qc}_quali'},inplace=True)

df_final_gp = pd.merge(df_training_overview,df_quali_overview,on=['Driver','Team','GP','Rule_Era'],how='left')
df_final_gp['Season'] = 2018
df_final_gp

In [ ]:
test = [
            '..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv',
            '..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv',
            '..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv'
        ]
test[1]

In [ ]:
def practice_quali(
        season=2018,
        gp='Australia',
        timing_paths=[
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1.csv',
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2.csv',
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3.csv',
            r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying.csv'
        ],
        weather_paths=[
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 1-weather.csv',
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 2-weather.csv',
            r'..\data\raw\2018\Australian GP\Practice\2018-Australian Grand Prix-Practice 3-weather.csv',
            r'..\data\raw\2018\Australian GP\Qualifying\2018-Australian Grand Prix-Qualifying-weather.csv'
        ]
):
    #practice 1, 2 and 3
    practice_1 = timing_paths[0]
    practice_1_weather = weather_paths[0]

    df_p1 = pd.read_csv(practice_1).iloc[:,1:]
    df_p1['Season'] = season
    df_p1['Rule_Era'] = df_p1.apply(f1_rule_era,axis=1)
    df_p1['Abandoned_Lap'] = df_p1.apply(abandoned_lap,axis=1)
    df_p1['GP'] = gp
    df_p1['Time'] = pd.to_timedelta(df_p1['Time'])
    df_p1['Time_Minutes'] = np.ceil(df_p1['Time'].dt.total_seconds()/60)
    df_p1_weather = pd.read_csv(practice_1_weather).iloc[:,1:]
    df_p1_weather['Time'] = pd.to_timedelta(df_p1_weather['Time'])
    df_p1_weather['Time_Minutes'] = np.ceil(df_p1_weather['Time'].dt.total_seconds()/60)
    df_p1_weather['Rainfall'] = df_p1_weather['Rainfall'].apply(lambda x: 1 if x else 0)
    df_p1_weather.drop(['Time'],axis=1,inplace=True)
    df_p1 = pd.merge(df_p1,df_p1_weather,on='Time_Minutes',how='inner')
    df_p1 = df_p1[df_p1['Abandoned_Lap'].eq(0)]


    practice_2 = timing_paths[1]
    practice_2_weather = weather_paths[1]

    df_p2 = pd.read_csv(practice_2).iloc[:,1:]
    df_p2['Season'] = season
    df_p2['Rule_Era'] = df_p2.apply(f1_rule_era,axis=1)
    df_p2['Abandoned_Lap'] = df_p2.apply(abandoned_lap,axis=1)
    df_p2['GP'] = gp
    df_p2['Time'] = pd.to_timedelta(df_p2['Time'])
    df_p2['Time_Minutes'] = np.ceil(df_p2['Time'].dt.total_seconds()/60)
    df_p2_weather = pd.read_csv(practice_2_weather).iloc[:,1:]
    df_p2_weather['Time'] = pd.to_timedelta(df_p2_weather['Time'])
    df_p2_weather['Time_Minutes'] = np.ceil(df_p2_weather['Time'].dt.total_seconds()/60)
    df_p2_weather['Rainfall'] = df_p2_weather['Rainfall'].apply(lambda x: 1 if x else 0)
    df_p2_weather.drop(['Time'],axis=1,inplace=True)
    df_p2 = pd.merge(df_p2,df_p2_weather,on='Time_Minutes',how='inner')
    df_p2 = df_p2[df_p2['Abandoned_Lap'].eq(0)]


    practice_3 = timing_paths[2]
    practice_3_weather = weather_paths[2]

    df_p3 = pd.read_csv(practice_3).iloc[:,1:]
    df_p3['Season'] = season
    df_p3['Rule_Era'] = df_p3.apply(f1_rule_era,axis=1)
    df_p3['Abandoned_Lap'] = df_p3.apply(abandoned_lap,axis=1)
    df_p3['GP'] = gp
    df_p3['Time'] = pd.to_timedelta(df_p3['Time'])
    df_p3['Time_Minutes'] = np.ceil(df_p3['Time'].dt.total_seconds()/60)
    df_p3_weather = pd.read_csv(practice_3_weather).iloc[:,1:]
    df_p3_weather['Time'] = pd.to_timedelta(df_p3_weather['Time'])
    df_p3_weather['Time_Minutes'] = np.ceil(df_p3_weather['Time'].dt.total_seconds()/60)
    df_p3_weather['Rainfall'] = df_p3_weather['Rainfall'].apply(lambda x: 1 if x else 0)
    df_p3_weather.drop(['Time'],axis=1,inplace=True)
    df_p3 = pd.merge(df_p3,df_p3_weather,on='Time_Minutes',how='inner')
    df_p3 = df_p3[df_p3['Abandoned_Lap'].eq(0)]

    combined = df_p1
    for d in [df_p2,df_p3]:
        combined = pd.concat([combined,d],axis=0)
    master_driver = combined[['Driver','Team','GP']].value_counts().to_frame().reset_index().iloc[:,:-1]

    #df_training_overview = pd.DataFrame()

    #practice 1
    training_1_time_w_diff = df_p1.groupby(
        #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(training_1_time_w_diff.columns)
    loc_cols = list(training_1_time_w_diff.columns)
    loc_cols.append('Compound')
    training_1_time_w_diff['LapTimeDiff'] = training_1_time_w_diff['laptime_sum_sectortimes'] - training_1_time_w_diff['laptime_sum_sectortimes'].min()
    df_p1.loc[:,loc_cols]
    p1_fast_time = pd.merge(training_1_time_w_diff,df_p1.loc[:,loc_cols],on=merge_cols,how='left')
    fastest_lap_idx = p1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    p1_fast_time = p1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    p1_fast_time = pd.merge(master_driver,p1_fast_time,on=['Driver','Team','GP'],how='left')
    group_col = ['Driver','Team','GP','Rule_Era']
    practice_col = [c for c in p1_fast_time.columns if c not in group_col]
    for pc in practice_col:
        p1_fast_time.rename(columns={pc:f'{pc}_p1'},inplace=True)
    p1_fast_time = p1_fast_time.sort_values('laptime_sum_sectortimes_p1',ascending=True).reset_index(drop=True)

    #practice 2
    training_2_time_w_diff = df_p2.groupby(
        #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(training_2_time_w_diff.columns)
    loc_cols = list(training_2_time_w_diff.columns)
    loc_cols.append('Compound')
    training_2_time_w_diff['LapTimeDiff'] = training_2_time_w_diff['laptime_sum_sectortimes'] - training_2_time_w_diff['laptime_sum_sectortimes'].min()
    df_p2.loc[:,loc_cols]
    p2_fast_time = pd.merge(training_2_time_w_diff,df_p2.loc[:,loc_cols],on=merge_cols,how='left')
    fastest_lap_idx = p2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    p2_fast_time = p2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    p2_fast_time = pd.merge(master_driver,p2_fast_time,on=['Driver','Team','GP'],how='left')
    group_col = ['Driver','Team','GP','Rule_Era']
    practice_col = [c for c in p2_fast_time.columns if c not in group_col]
    for pc in practice_col:
        p2_fast_time.rename(columns={pc:f'{pc}_p2'},inplace=True)
    p2_fast_time = p2_fast_time.sort_values('laptime_sum_sectortimes_p2',ascending=True).reset_index(drop=True)

    #practice 3
    training_3_time_w_diff = df_p3.groupby(
        #['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp','WindDirection','WindSpeed']
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(training_3_time_w_diff.columns)
    loc_cols = list(training_3_time_w_diff.columns)
    loc_cols.append('Compound')
    training_3_time_w_diff['LapTimeDiff'] = training_3_time_w_diff['laptime_sum_sectortimes'] - training_3_time_w_diff['laptime_sum_sectortimes'].min()
    df_p3.loc[:,loc_cols]
    p3_fast_time = pd.merge(training_3_time_w_diff,df_p3.loc[:,loc_cols],on=merge_cols,how='left')
    if p3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
        p3_fast_time['laptime_sum_sectortimes'] = p3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
        p3_fast_time['LapTimeDiff_P3'] = p3_fast_time['LapTimeDiff_P3'].fillna(0.0)
        p3_fast_time['Compound'] = p3_fast_time['Compound'].fillna('No Tyres Used')
    fastest_lap_idx = p3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    p3_fast_time = p3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    p3_fast_time = pd.merge(master_driver,p3_fast_time,on=['Driver','Team','GP'],how='left')
    group_col = ['Driver','Team','GP','Rule_Era']
    practice_col = [c for c in p3_fast_time.columns if c not in group_col]
    for pc in practice_col:
        p3_fast_time.rename(columns={pc:f'{pc}_p3'},inplace=True)
    p3_fast_time = p3_fast_time.sort_values('laptime_sum_sectortimes_p3',ascending=True).reset_index(drop=True)

    #practice overview
    trainings = [p2_fast_time,p3_fast_time]
    df_training_overview = p1_fast_time.copy()
    for t in trainings:
        df_training_overview = pd.merge(df_training_overview,t,on=['Driver','Team','GP','Rule_Era'],how='left')


    #Q1, Q2 and Q3
    #combining laptime with weather data
    quali = timing_paths[3]
    quali_weather = weather_paths[3]

    df_quali = pd.read_csv(quali).iloc[:,1:]
    df_quali['Season'] = season
    df_quali['Rule_Era'] = df_quali.apply(f1_rule_era,axis=1)
    df_quali['Abandoned_Lap'] = df_quali.apply(abandoned_lap,axis=1)
    df_quali['GP'] = gp
    df_quali['Time'] = pd.to_timedelta(df_quali['Time'])
    df_quali['Time_Minutes'] = np.ceil(df_quali['Time'].dt.total_seconds()/60)
    df_quali_weather = pd.read_csv(quali_weather).iloc[:,1:]
    df_quali_weather['Time'] = pd.to_timedelta(df_quali_weather['Time'])
    df_quali_weather['Time_Minutes'] = np.ceil(df_quali_weather['Time'].dt.total_seconds()/60)
    df_quali_weather['Rainfall'] = df_quali_weather['Rainfall'].apply(lambda x: 1 if x else 0)
    df_quali_weather.drop(['Time'],axis=1,inplace=True)
    df_quali = pd.merge(df_quali,df_quali_weather,on='Time_Minutes',how='inner')
    df_quali = df_quali[df_quali['Abandoned_Lap'].eq(0)]


    #creating the code block which allows us to divide into Q1, Q2 and Q3
    quali_time_df = df_quali['Time'].to_frame().sort_values('Time',ascending=True).reset_index(drop=True)
    quali_time_df['Time_Diff'] = (quali_time_df['Time'].dt.total_seconds()/60).diff()
    quali_time_df['Time_Minutes'] = np.ceil(quali_time_df['Time'].dt.total_seconds()/60)
    quali_time_df[quali_time_df['Time_Diff']>=5]
    #cutoff time for each qualifiying session
    quali_session_idx = [tv-1 for tv in quali_time_df[quali_time_df['Time_Diff']>=5].index]
    df_quali_session = quali_time_df.iloc[quali_session_idx]['Time_Minutes'].to_frame().reset_index(drop=True)
    df_quali_session.iloc[-1] = quali_time_df['Time_Minutes'].max()
    df_quali_session['Quali_Session'] = np.array(['Q1','Q2','Q3'])


    #Q1 laptimes
    df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][0])]
    quali_1_time_w_diff = df_q1.groupby(
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(quali_1_time_w_diff.columns)
    loc_cols = list(quali_1_time_w_diff.columns)
    loc_cols.append('Compound')
    quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
    df_q1.loc[:,loc_cols]
    q1_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
    if q1_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
        q1_fast_time['laptime_sum_sectortimes'] = q1_fast_time['laptime_sum_sectortimes'].fillna(0.0)
        q1_fast_time['LapTimeDiff'] = q1_fast_time['LapTimeDiff'].fillna(0.0)
        q1_fast_time['Compound'] = q1_fast_time['Compound'].fillna('No Tyres Used')
    fastest_lap_idx = q1_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    q1_fast_time = q1_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    q1_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][0]
    q1_fast_time = q1_fast_time.iloc[-5:,:]


    #Q2 laptimes
    df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][1])]
    quali_1_time_w_diff = df_q1.groupby(
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(quali_1_time_w_diff.columns)
    loc_cols = list(quali_1_time_w_diff.columns)
    loc_cols.append('Compound')
    quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
    df_q1.loc[:,loc_cols]
    q2_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
    if q2_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
        q2_fast_time['laptime_sum_sectortimes'] = q2_fast_time['laptime_sum_sectortimes'].fillna(0.0)
        q2_fast_time['LapTimeDiff'] = q2_fast_time['LapTimeDiff'].fillna(0.0)
        q2_fast_time['Compound'] = q2_fast_time['Compound'].fillna('No Tyres Used')
    q2_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][1]
    fastest_lap_idx = q2_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    q2_fast_time = q2_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    #here we drop the drivers from q1
    drivers_dropped_q1 = list(q1_fast_time['Driver'])
    q2_fast_time = q2_fast_time[~q2_fast_time['Driver'].isin(drivers_dropped_q1)]
    q2_fast_time = q2_fast_time.iloc[-5:,:]


    #Q3 laptimes
    df_q1 = df_quali[df_quali['Time']<=pd.Timedelta(minutes=df_quali_session['Time_Minutes'][2])]
    quali_1_time_w_diff = df_q1.groupby(
        ['Driver','Team','GP','Rule_Era','AirTemp','Humidity','Pressure','Rainfall','TrackTemp']
    )['laptime_sum_sectortimes'].min().to_frame().reset_index().sort_values('laptime_sum_sectortimes',ascending=True)
    merge_cols = list(quali_1_time_w_diff.columns)
    loc_cols = list(quali_1_time_w_diff.columns)
    loc_cols.append('Compound')
    quali_1_time_w_diff['LapTimeDiff'] = quali_1_time_w_diff['laptime_sum_sectortimes'] - quali_1_time_w_diff['laptime_sum_sectortimes'].min()
    df_q1.loc[:,loc_cols]
    q3_fast_time = pd.merge(quali_1_time_w_diff,df_q1.loc[:,loc_cols],on=merge_cols,how='left')
    if q3_fast_time['laptime_sum_sectortimes'].isna().sum().item() > 0:
        q3_fast_time['laptime_sum_sectortimes'] = q3_fast_time['laptime_sum_sectortimes'].fillna(0.0)
        q3_fast_time['LapTimeDiff'] = q3_fast_time['LapTimeDiff'].fillna(0.0)
        q3_fast_time['Compound'] = q3_fast_time['Compound'].fillna('No Tyres Used')
    q3_fast_time['Quali-Session'] = df_quali_session['Quali_Session'][2]
    fastest_lap_idx = q3_fast_time.groupby(['Driver','Team'])['laptime_sum_sectortimes'].idxmin()
    q3_fast_time = q3_fast_time.iloc[fastest_lap_idx].sort_values('laptime_sum_sectortimes',ascending=True).reset_index(drop=True)
    #here we drop the drivers from q1 and q2
    drivers_dropped_q1_a_q2 = drivers_dropped_q1 + list(q2_fast_time['Driver'])
    q3_fast_time = q3_fast_time[~q3_fast_time['Driver'].isin(drivers_dropped_q1_a_q2)]


    #total qualifiying overview
    df_quali_overview = pd.DataFrame()
    quali_sessions = [q1_fast_time,q2_fast_time,q3_fast_time]
    for q in quali_sessions:
        df_quali_overview = pd.concat([df_quali_overview,q],axis=0)
    df_quali_overview = df_quali_overview.sort_index()
    group_col = ['Driver','Team','GP','Rule_Era','Quali-Session']
    quali_col = [c for c in df_quali_overview.columns if c not in group_col]
    for qc in quali_col:
        df_quali_overview.rename(columns={qc:f'{qc}_quali'},inplace=True)

    df_final_gp = pd.merge(df_training_overview,df_quali_overview,on=['Driver','Team','GP','Rule_Era'],how='left')
    df_final_gp['Season'] = season
    return df_final_gp


In [ ]:
test = practice_quali()
test

In [ ]:
test_2 = practice_quali(
    season=2018,
    gp='China GP',
    timing_paths=[
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 1.csv',
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 2.csv',
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 3.csv',
        r'..\data\raw\2018\Chinese GP\Qualifying\2018-Chinese Grand Prix-Qualifying.csv'
    ],
    weather_paths=[
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 1-weather.csv',
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 2-weather.csv',
        r'..\data\raw\2018\Chinese GP\Practice\2018-Chinese Grand Prix-Practice 3-weather.csv',
        r'..\data\raw\2018\Chinese GP\Qualifying\2018-Chinese Grand Prix-Qualifying-weather.csv'
    ]
)
ttest_2 = test_2.sort_values('laptime_sum_sectortimes_quali',ascending=True)
ttest_2[['Driver','Team','laptime_sum_sectortimes_quali']]
test_2

In [ ]:
test_2 = practice_quali(
    season=2018,
    gp='China GP',
    timing_paths=[
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 1.csv',
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 2.csv',
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 3.csv',
        r'..\data\raw\2018\Bahrain GP\Qualifying\2018-Bahrain Grand Prix-Qualifying.csv'
    ],
    weather_paths=[
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 1-weather.csv',
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 2-weather.csv',
        r'..\data\raw\2018\Bahrain GP\Practice\2018-Bahrain Grand Prix-Practice 3-weather.csv',
        r'..\data\raw\2018\Bahrain GP\Qualifying\2018-Bahrain Grand Prix-Qualifying-weather.csv'
    ]
)
ttest_2 = test_2.sort_values('laptime_sum_sectortimes_quali',ascending=True)
ttest_2[['Driver','Team','laptime_sum_sectortimes_quali']]
test_2

# Lopping Multipel Datasets

In [ ]:
path_list=[]
data_path = Path(r'..\data\raw\2018')

for file in data_path.iterdir():
    if file.is_dir():
        #gp=file.split('\')
        path_list.append(file.name)
#full list of all sub-folders/GPs
path_list = path_list[1:]

folder_value = path_list
dataset_value = [dv.split(' G')[0] for dv in path_list]

for i in range(0,len(folder_value)):
    print(f'{dataset_value[i]}')

In [ ]:
path_list=[]
data_path = Path(r'..\data\raw\2022')

for file in data_path.iterdir():
    if file.is_dir():
        #gp=file.split('\')
        path_list.append(file.name)
#full list of all sub-folders/GPs
path_list = path_list[1:]

folder_value = path_list
dataset_value = [dv.split(' G')[0] for dv in path_list]

for i in range(0,len(folder_value)):
    print(f'{dataset_value[i]}')

In [ ]:
[y for y in range(2018,2027)]

In [ ]:
[rf"..\data\raw\{i}" for i in range(2018,2027)]

In [ ]:
#next step, intorduce an if clause, with season i, because the name of the sprint session changes between the seasons
df_2018_2026 = pd.DataFrame()

for i, p in zip([y for y in range(2018,2027)], [rf"..\data\raw\{i}" for i in range(2018,2027)]):
    data_path = Path(p)

    path_list = [f.name for f in data_path.iterdir() if f.is_dir()]
    path_list = path_list[1:]   # if needed

    if i == 2020:
        path_list = [x for x in path_list if x != 'Eifel GP']

    for folder_loop_value in path_list:
        base = data_path / folder_loop_value
        race_name_for_file = folder_loop_value.replace(" GP", "")

        prac_1 = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 1.csv"
        prac_2 = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 2.csv"
        prac_3 = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 3.csv"
        quali = base / "Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Qualifying.csv"
        if i < 2025:
            sprint_quali = base / "Sprint-Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Sprint Shootout.csv"
        elif i >= 2025:
            sprint_quali = base / "Sprint-Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Sprint Qualifying.csv"

        prac_1_weather = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 1-weather.csv"
        prac_2_weather = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 2-weather.csv"
        prac_3_weather = base / "Practice" / f"{i}-{race_name_for_file} Grand Prix-Practice 3-weather.csv"
        quali_weather = base / "Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Qualifying-weather.csv"
        if i < 2025:
            sprint_quali_weather = base / "Sprint-Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Sprint Shootout-weather.csv"
        elif i >= 2025:
            sprint_quali_weather = base / "Sprint-Qualifying" / f"{i}-{race_name_for_file} Grand Prix-Sprint Qualifying-weather.csv"

        df_loop = practice_quali_new(
            season=i,
            gp=folder_loop_value,
            timing_paths=[prac_1, prac_2, prac_3, quali,sprint_quali],
            weather_paths=[prac_1_weather, prac_2_weather, prac_3_weather, quali_weather, sprint_quali_weather]
        )

        df_2018_2026 = pd.concat([df_2018_2026, df_loop], ignore_index=True)

df_2018_2026

### Test

In [ ]:
n_count = 0
check_list = []
while n_count < 10:
    season_ = random.choice(range(2018,2027))
    gp_ = random.choice(df_2018_2026[df_2018_2026['Season'].eq(season_)]['GP'].unique())
    check_list.append((season_,gp_))
    n_count+=1
print(check_list)

In [ ]:
#problems (2023, Beligan GP),(2024, Hunagrian GP)

s = 2025
gp = 'Spanish GP'

df_2018_2026[(df_2018_2026['Season'].eq(s)) & (df_2018_2026['GP'].eq(gp))][[
    'Driver','Team','GP','Sprint-Session','laptime_sum_sectortimes_sprint_quali','LapTimeDiff_sprint_quali','Session','laptime_sum_sectortimes_quali','LapTimeDiff_quali',
    'Sprint_Race_Era','Sprint_Weekend']]

# Logic Tests

In [ ]:
##look into Belgian GP, Hungarian GP, Japan GP => due to rain, time mix up
# ##=> checking for rain column, when ticked and these laps fall into later qualifying periods, we should use these

for i in range(2018,2027):
    for gp in df_2018_2026[df_2018_2026['Season'].eq(i)]['GP'].unique():
        top_10_output = df_2018_2026[(df_2018_2026['GP'].eq(gp)) & (df_2018_2026['Season'].eq(i))][['Driver','laptime_sum_sectortimes_quali','LapTimeDiff_quali']].nsmallest(10,'laptime_sum_sectortimes_quali')
        print(f'\n{gp}-{i}')
        print(f'{top_10_output}\n')

In [ ]:
df_2023_bel = pd.DataFrame()

prac_1 = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 1.csv")
prac_2 = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 2.csv")
prac_3 = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 3.csv")
quali = Path(r"..\data\raw\2023\Belgian GP\Qualifying\2023-Belgian Grand Prix-Qualifying.csv")
sprint_quali = Path(r"..\data\raw\2023\Belgian GP\Sprint-Qualifying\2023-Belgian Grand Prix-Sprint Qualifying.csv")

prac_1_weather = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 1-weather.csv")
prac_2_weather = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 2-weather.csv")
prac_3_weather = Path(r"..\data\raw\2023\Belgian GP\Practice\2023-Belgian Grand Prix-Practice 3-weather.csv")
quali_weather = Path(r"..\data\raw\2023\Belgian GP\Qualifying\2023-Belgian Grand Prix-Qualifying-weather.csv")
sprint_quali_weather = Path(r"..\data\raw\2023\Belgian GP\Qualifying\2023-Belgian Grand Prix-Qualifying-weather.csv")

df_loop = practice_quali_new(
    season=2023,
    gp='Belgian GP',
    timing_paths=[prac_1, prac_2, prac_3, quali,sprint_quali],
    weather_paths=[prac_1_weather, prac_2_weather, prac_3_weather, quali_weather,sprint_quali_weather]
)

df_2023_bel = pd.concat([df_2023_bel, df_loop], ignore_index=True)
df_2023_bel

# Circut Information

In this section, we will add circut information. The data comes from a Excel file which was assambled using different websites:

- https://en.wikipedia.org/wiki/List_of_Formula_One_circuits
- https://www.formula1.com/en/racing/ => from 2018 - 2026
- https://formula-timer.com/circuit
- https://oversteer48.com/
- https://www.statsf1.com/en/default.aspx
- https://medium.com/%40kenneth.agregaard.jensen/analyzing-formula-1-data-part-3-6b6ebc8a7714

The excel file contains an explanation of following terms/columns:
- Turn density => Calculated as turns divided by last length used (turns per km)
- Complexity label => Very twisty >= 3.80 turns/km; Twisty 3.30–3.79; Balanced 2.90–3.29; Flowing < 2.90

In [ ]:
df_quali = pd.read_csv(r'..\data\processed\df_2018_2026_quali_output.csv')

df_circut=pd.read_excel(r'..\data\raw\f1_circuits_2018_2026_extended.xlsx').iloc[:,:-4]
df_circut['Circut_length'] = df_circut['Circut_length'].apply(lambda x: round(float(x.split(' ')[0]),2))
df_circut['Pace_profile'] = df_circut['Pace_profile'].apply(lambda x: x.split('/')[0] if x.split('/') else x)
df_circut.drop(['Confidence'],axis=1,inplace=True)
#adding 
df_circut_adjust = df_circut[df_circut['Location'].isin(['Silverstone','Sakhir','Spielberg'])].replace({
    'Bahrain Grand Prix; Sakhir Grand Prix':'Sakhir Grand Prix',
    'Austrian Grand Prix; Styrian Grand Prix':'Styrian Grand Prix',
    'British Grand Prix; 70th Anniversary Grand Prix':'70th Anniversary Grand Prix'
    }).copy()
df_circut = pd.concat([df_circut,df_circut_adjust],axis=0)
df_circut.rename(columns={'Grand_Prix(es)':'GP'},inplace=True)
df_circut.replace({
    'Bahrain Grand Prix; Sakhir Grand Prix':'Bahrain Grand Prix',
    'Austrian Grand Prix; Styrian Grand Prix':'Austrian Grand Prix',
    'British Grand Prix; 70th Anniversary Grand Prix':'British Grand Prix',
    'Italian Grand Prix; San Marino Grand Prix; Emilia-Romagna Grand Prix':'Emilia Romagna Grand Prix',
    'Brazilian Grand Prix; São Paulo Grand Prix':'Brazilian Grand Prix',
    'Mexican Grand Prix; Mexico City Grand Prix':'Mexican Grand Prix',
    'European Grand Prix; Azerbaijan Grand Prix':'Azerbaijan Grand Prix',
    'Spanish Grand Prix':'Madrid Grand Prix',
    'Spanish Grand Prix; Barcelona-Catalunya Grand Prix':'Spanish Grand Prix',
    'German Grand Prix; European Grand Prix; Luxembourg Grand Prix; Eifel Grand Prix':'Eifel Grand Prix',
    },inplace=True)
df_circut['GP'] = df_circut['GP'].apply(lambda x: x.replace('Grand Prix','GP'))
df_circut = df_circut[['GP','Type','Direction','Circut_length','Turns','Pace_profile',
           'Flat_out_run','Slow_turns','Medium_turns','High_speed_turns','Turn_density','Complexity_label']]
#df_circut.to_csv('..\data\processed\cleaned_f1_circut.csv',index=False)
df_quali = pd.merge(df_quali,df_circut,how='left',on='GP')
df_quali.head()

In [ ]:
df_quali[df_quali['GP'].eq('Australian GP')]